In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import DataLoader, Dataset
from PIL import Image

# Chemins des dossiers et fichiers
ref_image_dir = "./data/DAM/"
test_image_dir = "./data/test_image_headmind/"
product_list_path = "./data/product_list.csv"

# Charger le fichier CSV
df = pd.read_csv(product_list_path)

# Lister les images réellement présentes dans le dossier DAM
dam_images = set(os.listdir(ref_image_dir))  # Liste des fichiers dans DAM

# Ajouter l'extension .jpeg aux MMC pour comparer
df["MMC_with_extension"] = df["MMC"] + ".jpeg"

# Filtrer pour ne garder que les MMC présents dans le dossier DAM
df_valid = df[df["MMC_with_extension"].isin(dam_images)]

# Mettre à jour les chemins des fichiers pour correspondre aux fichiers DAM
df_valid["MMC_path"] = df_valid["MMC_with_extension"].apply(
    lambda x: os.path.join(ref_image_dir, x)
)

# Résumé des données
print(f"Nombre total d'images dans DAM : {len(dam_images)}")
print(f"Nombre d'images associées à des entrées CSV : {len(df_valid)}")

# Préparer les classes et le mapping
classes = df_valid["Product_BusinessUnitDesc"].unique()
class_to_index = {cls: idx for idx, cls in enumerate(classes)}
index_to_class = {idx: cls for cls, idx in class_to_index.items()}
df_valid["class_id"] = df_valid["Product_BusinessUnitDesc"].map(class_to_index)


# Custom Dataset class
class ImageDataset(Dataset):
    def __init__(self, image_paths, labels=None, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"Image not found at path: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform:
            image = self.transform(image)
        if self.labels is not None:
            label = self.labels[idx]
            return image, label
        return image


# Data transformations
transform = transforms.Compose(
    [
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

# Préparer les données d'entraînement
image_paths = df_valid["MMC_path"].tolist()
labels = df_valid["class_id"].tolist()
train_dataset = ImageDataset(image_paths, labels, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)

# Charger un modèle pré-entraîné
model = models.efficientnet_b0(pretrained=True)
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, len(classes))
model = model.to("cuda" if torch.cuda.is_available() else "cpu")

# Définir la fonction de perte et l'optimiseur
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


# Fonction d'entraînement
def train_model(model, dataloader, criterion, optimizer, num_epochs=10):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
        epoch_loss = running_loss / len(dataloader.dataset)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")


# Entraîner le modèle
train_model(model, train_loader, criterion, optimizer, num_epochs=10)

# Générer les embeddings pour les images DAM
model.eval()
dam_embeddings = []
dam_ids = []
with torch.no_grad():
    for images, labels in train_loader:
        images = images.to("cuda" if torch.cuda.is_available() else "cpu")
        embeddings = model(images).cpu().numpy()
        dam_embeddings.append(embeddings)
        dam_ids.extend(labels.numpy())

dam_embeddings = np.vstack(dam_embeddings)


# Fonction pour prédire les images de test et associer l'image la plus proche
def predict_and_find_closest(model, test_image_dir, transform, dam_embeddings, dam_ids):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.eval()
    results = []
    test_images = [
        os.path.join(test_image_dir, img)
        for img in os.listdir(test_image_dir)
        if img.lower().endswith((".jpeg", ".jpg", ".png"))
    ]

    with torch.no_grad():
        for img_path in test_images:
            img = Image.open(img_path).convert("RGB")
            img = transform(np.array(img)).unsqueeze(0).to(device)

            # Extraire les embeddings pour l'image de test
            test_embedding = model(img).cpu().numpy()

            # Trouver l'image d'entraînement la plus proche
            distances = np.linalg.norm(dam_embeddings - test_embedding, axis=1)
            closest_idx = np.argmin(distances)

            closest_id = dam_ids[closest_idx]
            results.append(
                {
                    "Test Image": os.path.basename(img_path),
                    "Closest Train ID": closest_id,
                }
            )

    return pd.DataFrame(results)


# Prédire et sauvegarder les résultats
results_df = predict_and_find_closest(
    model, test_image_dir, transform, dam_embeddings, labels
)
results_df.to_csv("./classification_with_closest.csv", index=False)
print("Results saved to classification_with_closest.csv")


C:\Users\Gabriel\AppData\Local\Temp\ipykernel_25748\35859158.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_valid["MMC_path"] = df_valid["MMC_with_extension"].apply(
C:\Users\Gabriel\AppData\Local\Temp\ipykernel_25748\35859158.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_valid["class_id"] = df_valid["Product_BusinessUnitDesc"].map(class_to_index)
c:\Users\Gabriel\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretra

Nombre total d'images dans DAM : 2766
Nombre d'images associées à des entrées CSV : 2766
Epoch 1/10, Loss: 1.4555
Epoch 2/10, Loss: 1.2957
